# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

# Read from Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info_raw")
display(df)

# Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

### Normalization

In [0]:
df = (
  df
    .withColumn(
      "cst_marital_status",
      F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
       .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
       .otherwise("n/a")
    )
    .withColumn(
      "cst_gndr",
      F.when(F.upper(col("cst_gndr")) == "M", "Male")
       .when(F.upper(col("cst_gndr")) == "F", "Female")
       .otherwise("n/a")
    )
)

### Remove records with missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

### Casting Date

In [0]:
df = df.withColumn("cst_create_date", col("cst_create_date").cast(DateType()))

### Rename Columns

In [0]:
RENAME_MAP = {
  "cst_id": "costumer_id",
  "cst_key": "costumer_key",
  "cst_firstname": "costumer_first_name",
  "cst_lastname": "costumer_last_name",
  "cst_marital_status": "costumer_marital_status",
  "cst_gndr": "costumer_gender",
  "cst_create_date": "costumer_create_date"
}

for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)

# Check Dataframe

In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

# Check Silver Table

In [0]:
%sql
SELECT *
FROM silver.crm_customers